# Phase 03 — Literature & Baseline Research
## CodeMentor-LLM
Testing Mistral-7B-Instruct-v0.3 base model on 10 coding questions
before any fine-tuning to justify our project.

#  Libraries

In [1]:
!pip install -q transformers==4.49.0 bitsandbytes==0.45.3 accelerate==1.5.1 peft==0.14.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 119.6 MB/s eta 0:00:00


# Checking GPU

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

CUDA available: True
GPU: Tesla T4
Memory: 14.56 GB


# Login to HuggingFace

In [3]:
from huggingface_hub import login
login()

# Load Mistral-7B-Instruct-v0.3 in 4-bit

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully")
print(f"Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully
Memory footprint: 3.75 GB


# Create inference function

In [7]:
def generate_response(prompt, max_new_tokens=256):
    # Apply Mistral chat template
    messages = [{"role": "user", "content": prompt}]

    # Tokenize
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only new tokens
    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )
    return response

print("Inference function ready")

Inference function ready


# Run 10 coding questions against base model

In [8]:
# 10 coding questions to test base model
questions = [
    "Write a Python function to reverse a string.",
    "Explain what a decorator is in Python with an example.",
    "What is the difference between a list and a tuple in Python?",
    "Write a Python function to check if a number is prime.",
    "How do you handle exceptions in Python? Show an example.",
    "Write a SQL query to find the second highest salary from a table.",
    "Explain the concept of recursion with a Python example.",
    "What is the difference between == and is in Python?",
    "Write a Python function to find duplicates in a list.",
    "Explain what a REST API is in simple terms.",
]

# Run all 10 questions
results = []
for i, question in enumerate(questions):
    print(f"\n{'='*30}")
    print(f"Q{i+1}: {question}")
    print(f"{'='*30}")
    response = generate_response(question)
    print(f"A: {response}")
    results.append({"question": question, "response": response})

print("\nAll 10 questions completed")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Q1: Write a Python function to reverse a string.
A: Here is a simple Python function to reverse a string:

```python
def reverse_string(input_string):
    reversed_string = ''
    for i in range(len(input_string)-1, -1, -1):
        reversed_string += input_string[i]
    return reversed_string
```

You can use this function like this:

```python
print(reverse_string("Hello, World!"))  # Output: !dlroW ,olleH
```

Alternatively, you can also use Python's built-in `reversed()` function to reverse a string:

```python
def reverse_string_built_in(input_string):
    return ''.join(reversed(input_string))
```

This function does the same thing as the previous one, but uses a built-in function instead of a loop.

```python
print(reverse_string_built_in("Hello, World!"))  # Output: !dlroW ,olleH
```

Both functions work the same way,

Q2: Explain what a decorator is in Python with an example.
A: In Python, a decorator is a special kind of function that allows you to add new functionality to a

## Base Model Analysis

### What the base model does WELL:
- Answers basic coding questions correctly
- Formats code blocks properly
- Gives multiple approaches (Q1 — loop + built-in)
- Explains concepts clearly (Q2, Q3, Q7, Q8)
- Handles SQL questions (Q6)

### What the base model FAILS at:
- Responses get cut off mid-sentence (Q2, Q3, Q4, Q7)
- No consistent response structure
- No step-by-step teaching format
- Inconsistent response length
- Warning: attention mask not set — tokenization issue
- Not optimized specifically for coding instruction format

### Why Fine-Tuning is justified:
- Base model cuts off responses frequently
- No consistent teaching structure
- Fine-tuning on CodeAlpaca-20K will teach it:
  - Complete structured responses
  - Consistent coding instruction format
  - Better code explanation style

# Save results to JSON

In [9]:
import json

# Save baseline results
baseline_results = {
    "model": "mistralai/Mistral-7B-Instruct-v0.3",
    "type": "base_model_no_finetuning",
    "results": results,
    "observations": {
        "strengths": [
            "Answers basic coding questions correctly",
            "Formats code blocks properly",
            "Gives multiple approaches",
            "Explains concepts clearly"
        ],
        "weaknesses": [
            "Responses get cut off mid-sentence",
            "No consistent response structure",
            "No step-by-step teaching format",
            "Inconsistent response length",
            "Attention mask warning"
        ],
        "finetuning_justified": True
    }
}

# Save to file
with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)

print("Baseline results saved to baseline_results.json")

Baseline results saved to baseline_results.json
